Usar 6 modelos para la predicción de la calidad del aire

Comprobar archivos

In [1]:
import pandas as pd
df = pd.read_csv('../Data/datos_limpios.csv')
print(df.head())

              datetime  PM2_5_ugm3  PM10_ugm3    O3_ppm   NO2_ppm   SO2_ppm  \
0  2025-11-06 20:09:00        22.0       26.0  0.008659  0.010098  0.020990   
1  2025-11-06 20:10:00        21.0       24.0  0.011206  0.013286  0.022898   
2  2025-11-06 20:11:00        21.0       25.0  0.010188  0.011161  0.022898   
3  2025-11-06 20:12:00        20.0       23.0  0.010697  0.011692  0.022135   
4  2025-11-06 20:13:00        20.0       22.0  0.009678  0.011692  0.022517   

     CO_ppm  
0  1.225555  
1  1.126044  
2  1.073670  
3  1.087637  
4  1.042246  


In [2]:
df.columns.tolist()

['datetime',
 'PM2_5_ugm3',
 'PM10_ugm3',
 'O3_ppm',
 'NO2_ppm',
 'SO2_ppm',
 'CO_ppm']

In [3]:
# =============================================================================
# CELDA 1: IMPORTACIONES Y CONFIGURACIÓN INICIAL
# =============================================================================
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# Datos (asume que df ya está cargado con las columnas necesarias)
# df['datetime'], df['PM2.5'], df['PM10'], df['O3'], df['NO2'], df['SO2'], df['CO']

# 1. Convertir datetime y extraer features
df['datetime'] = pd.to_datetime(df['datetime'])
df['hour'] = df['datetime'].dt.hour
df['day_of_week'] = df['datetime'].dt.dayofweek
df['month'] = df['datetime'].dt.month
df['is_weekend'] = df['day_of_week'].isin([5, 6]).astype(int)

# 2. Features cíclicas
df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)
df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)

# 3. Configuración X y Y
Y = df['PM2_5_ugm3'].values
X = df[['PM10_ugm3', 'O3_ppm', 'NO2_ppm', 'SO2_ppm', 'CO_ppm', 
        'hour_sin', 'hour_cos', 'month_sin', 'month_cos', 
        'day_of_week', 'is_weekend']].values

# 4. Dividir train/test
X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=0.2, random_state=42
)

# Escalar datos (necesario para SVM, MLP, RNN)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"✅ Datos preparados: X_train={X_train.shape}, Y_train={Y_train.shape}")

✅ Datos preparados: X_train=(14559, 11), Y_train=(14559,)


Regresión Lineal OLS

In [4]:
# =============================================================================
# CELDA 2: MODELO 1 - REGRESIÓN LINEAL CON OLS
# =============================================================================
import statsmodels.api as sm

# Agregar intercepto (constante) para OLS
X_train_ols = sm.add_constant(X_train_scaled)
X_test_ols = sm.add_constant(X_test_scaled)

# Entrenar modelo OLS
modelo_ols = sm.OLS(Y_train, X_train_ols).fit()

# Predecir
Y_pred_ols = modelo_ols.predict(X_test_ols)

# Evaluar
mse_ols = mean_squared_error(Y_test, Y_pred_ols)
rmse_ols = np.sqrt(mse_ols)
mae_ols = mean_absolute_error(Y_test, Y_pred_ols)
r2_ols = r2_score(Y_test, Y_pred_ols)

print("="*60)
print("🔵 MODELO 1: Regresión Lineal OLS")
print("="*60)
print(f"  RMSE: {rmse_ols:.2f} μg/m³")
print(f"  MAE:  {mae_ols:.2f} μg/m³")
print(f"  R²:   {r2_ols:.4f}")
print(f"\n📈 Resumen del modelo:")
print(modelo_ols.summary())

🔵 MODELO 1: Regresión Lineal OLS
  RMSE: 1.07 μg/m³
  MAE:  0.32 μg/m³
  R²:   0.9920

📈 Resumen del modelo:
                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.947
Model:                            OLS   Adj. R-squared:                  0.947
Method:                 Least Squares   F-statistic:                 2.916e+04
Date:                Sat, 16 May 2026   Prob (F-statistic):               0.00
Time:                        16:02:34   Log-Likelihood:                -36168.
No. Observations:               14559   AIC:                         7.236e+04
Df Residuals:                   14549   BIC:                         7.243e+04
Df Model:                           9                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
----------------------

Regresión Lineal GLS

In [5]:
# =============================================================================
# CELDA 3: MODELO 2 - REGRESIÓN LINEAL CON GLS
# =============================================================================
from statsmodels.regression.linear_model import GLSAR

# GLS con estructura autoregresiva AR(1) para datos temporales
modelo_gls = GLSAR(Y_train, X_train_ols, rho=1)
modelo_gls_result = modelo_gls.iterative_fit(maxiter=10)

# Predecir
Y_pred_gls = modelo_gls_result.predict(X_test_ols)

# Evaluar
mse_gls = mean_squared_error(Y_test, Y_pred_gls)
rmse_gls = np.sqrt(mse_gls)
mae_gls = mean_absolute_error(Y_test, Y_pred_gls)
r2_gls = r2_score(Y_test, Y_pred_gls)

print("="*60)
print("🔵 MODELO 2: Regresión Lineal GLS (GLSAR AR(1))")
print("="*60)
print(f"  RMSE: {rmse_gls:.2f} μg/m³")
print(f"  MAE:  {mae_gls:.2f} μg/m³")
print(f"  R²:   {r2_gls:.4f}")
print(f"\n📈 Rho estimado (autocorrelación): {modelo_gls.rho}")
print(f"  Coeficientes: {modelo_gls_result.params}")

🔵 MODELO 2: Regresión Lineal GLS (GLSAR AR(1))
  RMSE: 1.07 μg/m³
  MAE:  0.32 μg/m³
  R²:   0.9920

📈 Rho estimado (autocorrelación): [0.00089766]
  Coeficientes: [ 1.22817685e+01  6.75550016e-02 -8.58683419e-03 -8.44612631e-02
 -2.32884306e-02 -4.86862185e-03 -1.46102469e-01 -9.94387429e+10
  6.78249642e+13 -7.20828562e-02  8.26433214e-02]


Support Vector Machine (SVM)

In [6]:
# =============================================================================
# CELDA 4: MODELO 3 - SVM (SUPPORT VECTOR MACHINE)
# =============================================================================
from sklearn.svm import SVR

# Entrenar SVM con kernel RBF (necesita datos escalados)
modelo_svm = SVR(kernel='rbf', C=100, gamma='scale', epsilon=0.1)
modelo_svm.fit(X_train_scaled, Y_train)

# Predecir
Y_pred_svm = modelo_svm.predict(X_test_scaled)

# Evaluar
mse_svm = mean_squared_error(Y_test, Y_pred_svm)
rmse_svm = np.sqrt(mse_svm)
mae_svm = mean_absolute_error(Y_test, Y_pred_svm)
r2_svm = r2_score(Y_test, Y_pred_svm)

print("="*60)
print("🟢 MODELO 3: SVM (SVR - RBF Kernel)")
print("="*60)
print(f"  RMSE: {rmse_svm:.2f} μg/m³")
print(f"  MAE:  {mae_svm:.2f} μg/m³")
print(f"  R²:   {r2_svm:.4f}")

🟢 MODELO 3: SVM (SVR - RBF Kernel)
  RMSE: 1.06 μg/m³
  MAE:  0.25 μg/m³
  R²:   0.9920


Random Forest

In [7]:
# =============================================================================
# CELDA 5: MODELO 4 - RANDOM FOREST
# =============================================================================
from sklearn.ensemble import RandomForestRegressor

# Entrenar Random Forest (NO necesita datos escalados)
modelo_rf = RandomForestRegressor(
    n_estimators=100,
    max_depth=15,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)
modelo_rf.fit(X_train, Y_train)

# Predecir
Y_pred_rf = modelo_rf.predict(X_test)

# Evaluar
mse_rf = mean_squared_error(Y_test, Y_pred_rf)
rmse_rf = np.sqrt(mse_rf)
mae_rf = mean_absolute_error(Y_test, Y_pred_rf)
r2_rf = r2_score(Y_test, Y_pred_rf)

# Importancia de features
importancia_rf = pd.DataFrame({
    'Feature': ['PM10', 'O3', 'NO2', 'SO2', 'CO', 
                'hour_sin', 'hour_cos', 'month_sin', 'month_cos', 
                'day_of_week', 'is_weekend'],
    'Importancia': modelo_rf.feature_importances_
}).sort_values('Importancia', ascending=False)

print("="*60)
print("🟠 MODELO 4: Random Forest")
print("="*60)
print(f"  RMSE: {rmse_rf:.2f} μg/m³")
print(f"  MAE:  {mae_rf:.2f} μg/m³")
print(f"  R²:   {r2_rf:.4f}")
print(f"\n📊 Importancia de Features:")
print(importancia_rf.to_string(index=False))

🟠 MODELO 4: Random Forest
  RMSE: 1.80 μg/m³
  MAE:  0.21 μg/m³
  R²:   0.9773

📊 Importancia de Features:
    Feature  Importancia
       PM10     0.969537
        NO2     0.009305
         CO     0.008507
   hour_cos     0.006787
         O3     0.002992
        SO2     0.002108
   hour_sin     0.000733
day_of_week     0.000024
 is_weekend     0.000008
  month_cos     0.000000
  month_sin     0.000000


Vanilla Recurrent Neural Network

In [10]:
# =============================================================================
# CELDA 6: MODELO 5 - PYTORCH VANILLA RECURRENT NEURAL NETWORK (RNN)
# =============================================================================
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# If you haven't already: create temporal sequences
def crear_secuencias(X, y, seq_length=24):
    X_seq, y_seq = [], []
    for i in range(len(X) - seq_length):
        X_seq.append(X[i:i+seq_length])
        y_seq.append(y[i+seq_length])
    return np.array(X_seq), np.array(y_seq)


seq_length = 24
X_scaled_total = scaler.fit_transform(X)  # assuming scaler is already defined
X_seq, Y_seq = crear_secuencias(X_scaled_total, Y, seq_length)

split_idx = int(len(X_seq) * 0.8)
X_seq_train, X_seq_test = X_seq[:split_idx], X_seq[split_idx:]
Y_seq_train, Y_seq_test = Y_seq[:split_idx], Y_seq[split_idx:]


print(f"📐 Secuencias: X_train={X_seq_train.shape}, X_test={X_seq_test.shape}")


# PyTorch RNN model (simple vanilla RNN)
class SimpleRNN(nn.Module):
    def __init__(self, input_size, hidden_size=50, output_size=1):
        super().__init__()
        self.rnn = nn.RNN(
            input_size=input_size,
            hidden_size=hidden_size,
            nonlinearity="tanh",
            batch_first=True,
        )
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        # x shape: (batch, seq_len, input_size)
        rnn_out, _ = self.rnn(x)  # rnn_out shape: (batch, seq_len, hidden_size)
        out = self.fc(rnn_out[:, -1, :])  # take last time step
        return out.squeeze(-1)  # (batch, 1) → (batch,)


# Data to PyTorch tensors
X_train_t = torch.tensor(X_seq_train, dtype=torch.float32).to(device)
Y_train_t = torch.tensor(Y_seq_train, dtype=torch.float32).to(device)

X_test_t = torch.tensor(X_seq_test, dtype=torch.float32).to(device)
Y_test_t = torch.tensor(Y_seq_test, dtype=torch.float32).to(device)


# Model, optimizer, loss
input_size = X_seq_train.shape[2]
model = SimpleRNN(input_size=input_size, hidden_size=50).to(device)

optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.MSELoss()


# Training loop
torch.manual_seed(42)
EPOCHS = 50
BATCH_SIZE = 32

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0

    for i in range(0, len(X_train_t), BATCH_SIZE):
        X_batch = X_train_t[i:i+BATCH_SIZE]
        Y_batch = Y_train_t[i:i+BATCH_SIZE]

        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, Y_batch)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    if (epoch + 1) % 10 == 0 or epoch == 0:
        print(f"Epoch {epoch+1:2d} | Train MSE: {running_loss / (len(X_train_t)//BATCH_SIZE + 1):.6f}")


# Predict
model.eval()
with torch.no_grad():
    Y_pred_rnn = model(X_test_t).cpu().numpy().flatten()


# Evaluar
mse_rnn = mean_squared_error(Y_seq_test, Y_pred_rnn)
rmse_rnn = np.sqrt(mse_rnn)
mae_rnn = mean_absolute_error(Y_seq_test, Y_pred_rnn)
r2_rnn = r2_score(Y_seq_test, Y_pred_rnn)


print("="*60)
print("🟣 MODELO 5: Vanilla RNN (PyTorch)")
print("="*60)
print(f"  RMSE: {rmse_rnn:.2f} μg/m³")
print(f"  MAE:  {mae_rnn:.2f} μg/m³")
print(f"  R²:   {r2_rnn:.4f}")

ModuleNotFoundError: No module named 'torch'

Multilayer Perceptron (MLP)

In [8]:
# =============================================================================
# CELDA 7: MODELO 6 - MLP (MULTI-LAYER PERCEPTRON)
# =============================================================================
from sklearn.neural_network import MLPRegressor

# Entrenar MLP (necesita datos escalados)
modelo_mlp = MLPRegressor(
    hidden_layer_sizes=(128, 64, 32),
    activation='relu',
    solver='adam',
    alpha=0.001,
    batch_size=32,
    learning_rate='adaptive',
    max_iter=500,
    random_state=42,
    verbose=False
)
modelo_mlp.fit(X_train_scaled, Y_train)

# Predecir
Y_pred_mlp = modelo_mlp.predict(X_test_scaled)

# Evaluar
mse_mlp = mean_squared_error(Y_test, Y_pred_mlp)
rmse_mlp = np.sqrt(mse_mlp)
mae_mlp = mean_absolute_error(Y_test, Y_pred_mlp)
r2_mlp = r2_score(Y_test, Y_pred_mlp)

print("="*60)
print("🔴 MODELO 6: MLP (Multi-Layer Perceptron)")
print("="*60)
print(f"  RMSE: {rmse_mlp:.2f} μg/m³")
print(f"  MAE:  {mae_mlp:.2f} μg/m³")
print(f"  R²:   {r2_mlp:.4f}")

🔴 MODELO 6: MLP (Multi-Layer Perceptron)
  RMSE: 1.30 μg/m³
  MAE:  0.55 μg/m³
  R²:   0.9880
